# Figure 3 — Double-KO interaction model on the Doxo1 score

For the perturbations shown in the Fig2 **e1** Doxo1-vs-pseudotime plot — those
**enriched in the differentiated states** (NEPC-A-1/-2 vs NTC; Fisher OR>1,
FDR<`ENR_FDR`) **+ NEUROG1+SIM1** (experimentally validated) — we decompose each
double KO's effect on the Doxo1 differentiation score with **plain statsmodels
OLS**, coefficients and p-values read straight off the fitted models. Fit on the
non-NE compartment (intermediate + differentiated), **per timepoint**, with **no
cell-state correction** (the composition-inclusive view).

**Two separate models per combo (g1, g2):**
- **Individual + interaction:** `Doxo1 ~ g1 + g2 + g1:g2` (NTC + cells with only
  g1/g2 guides) → `β_g1`, `β_g2`, `β_g1:g2` and their p-values.
- **Total effect:** `Doxo1 ~ combo` (NTC vs the double-KO cells) → the combo
  coefficient = total double-KO effect vs NTC, with its p-value.

All p-values are the two-sided Wald p from statsmodels; BH-FDR across combos per
effect. The heatmap shows the four columns (base g1, base g2, interaction, total).

In [ ]:
import _figutils as fu
import importlib; importlib.reload(fu)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

fu.set_theme()
adata = fu.load("processed")

ENR_FDR = 0.1            # differentiated-state enrichment cutoff (matches Fig2)
FDR = 0.1               # FDR cutoff for the heatmap stars
FORCE = "NEUROG1+SIM1"  # experimentally-validated combo, always shown

# Perturbations shown in Fig2 e1 = enriched in the differentiated states + NEUROG1+SIM1.
# Read from Fig2's saved enrichment table (run Fig2's panel-c/selection cell first).
def e1_combos(day):
    enr = pd.read_csv(fu.FIG_DIR / f"Fig2_state_enrichment_{day}.csv")
    diff = enr[(enr.state.isin(fu.DIFFERENTIATED_STATES)) &
               (enr.odds_ratio > 1) & (enr.fdr < ENR_FDR)]
    pset = set(diff["perturbation"]) | {FORCE}
    return [p for p in pset if "+" in p]        # only combos carry an interaction term

# Per-combo OLS: (1) Doxo1 ~ g1 + g2 + g1:g2 for the base + interaction effects,
# (2) a separate Doxo1 ~ combo model for the total effect. Plain statsmodels,
# coefficients/p-values read straight off; non-NE compartment, per timepoint.
tables_by_day = {}
for day in fu.TIMEPOINT_ORDER:
    tab = fu.interaction_doxo1_ols(adata, e1_combos(day), day=day, min_cells=10)
    tab = tab.sort_values("total_effect", ascending=False).reset_index(drop=True)
    tab.to_csv(fu.FIG_DIR / f"Fig3_interaction_{day}.csv", index=False)
    tables_by_day[day] = tab
    print(f"\n{day}: {tab['total_effect'].notna().sum()} combos")
    print(tab[["pair", "beta_g1", "pval_g1", "beta_g2", "pval_g2",
               "beta_interaction", "pval_interaction", "total_effect", "pval_total"]]
          .round(3).to_string(index=False))

## Four-column heatmap per timepoint

Rows = the e1 double KOs (ordered by total effect); columns = base effect of gene 1,
base effect of gene 2, the interaction, and the total double-KO effect. Each cell is
starred by **its own** BH-FDR (base/interaction from the `g1 + g2 + g1:g2` model, total
from the separate `combo`-vs-NTC model). NEUROG1+SIM1 is always included. The full
per-combo tables (all betas, p-values, FDRs) are saved as `Fig3_interaction_{day}.csv`.

In [ ]:
def _star(p):
    return "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < FDR else ""

def four_col_heatmap(tab, title, fname):
    d = tab[tab["total_effect"].notna()].sort_values("total_effect", ascending=False)
    if not len(d):
        print(f"  nothing to plot for {title}"); return
    d = d.set_index("pair")
    cols = ["beta_g1", "beta_g2", "beta_interaction", "total_effect"]
    fdr_cols = ["fdr_g1", "fdr_g2", "fdr_interaction", "fdr_total"]   # star each column by its own FDR
    colnames = ["Main\n(gene 1)", "Main\n(gene 2)", "Interaction", "Total\n(double KO)"]
    mat = d[cols].values
    annot = np.array([[f"{mat[i, j]:.3f}{_star(d[fdr_cols[j]].values[i])}"
                       for j in range(mat.shape[1])] for i in range(mat.shape[0])], dtype=object)
    vmax = np.nanmax(np.abs(mat)) or 1.0
    fig, ax = plt.subplots(figsize=(6, 6))
    sns.heatmap(pd.DataFrame(mat, index=d.index, columns=colnames), cmap="RdBu_r",
                center=0, vmin=-vmax, vmax=vmax, annot=annot, fmt="", linewidths=0.6,
                linecolor="white", cbar_kws={"label": "effect on Doxo1 score"}, ax=ax,
                annot_kws={"fontsize": 8})
    ax.axvline(3, color="black", lw=2)          # separate the total column
    ax.set_title(title + f"\n(* FDR<{FDR}, ** <0.01, *** <0.001; two-sided Wald)", fontsize=8)
    ax.set_ylabel("Double-KO pair"); ax.set_xlabel("")
    plt.yticks(rotation=0); plt.xticks(rotation=0)
    fu.savefig(fname, fig)

for day in fu.TIMEPOINT_ORDER:
    four_col_heatmap(tables_by_day[day],
                     f"Double-KO effects on Doxo1 — Fig2 e1 perturbations — {day}",
                     f"Fig3_interaction_heatmap_{day}")

## E — Perturbation → cell state → Doxo1: two routes to a higher Doxo1 score

The 6 cell states were defined from the **whole transcriptome**, independently of the
Doxo1 signature — so a perturbation can raise the Doxo1 score by two distinct routes,
each estimated with a **statsmodels regression** (fit fresh per displayed cell,
one-sided combo > NTC, BH-FDR across the displayed grid):
- **Composition shift** — **logistic** regression `I(cell in state) ~ combo`; the
  colour is the **β = log-odds** of a combo cell being in that state vs NTC.
- **Within-state effect** — **OLS** regression `Doxo1 ~ combo`; the colour is the
  **β = Δ Doxo1** (combo vs NTC) within that state.

`*` marks FDR<0.1 for exactly the cells shown; a cell that can't be fit (within-state
combo n<3, or logistic separation) is greyed and marked `·`.

Most e1 double KOs act by **composition** (significant positive log-odds in the
differentiated states, mostly NEPC-A-1). **NEUROG1+SIM1** (boxed) is the exception:
**no significant composition shift** in any state, but it **raises Doxo1 within
NEPC-A-2** (OLS β=+0.073, FDR=0.035 — the only significant within-state cell) — exactly
why it was added to the e1 set despite not being a state-shifter. Per-cell stats
(betas, p-values, FDRs) are saved to `Fig3_state_vs_doxo1_{day}.csv`.

In [ ]:
# Both panels are statsmodels regression models, fit fresh per displayed (combo x
# state) cell, one-sided (combo > NTC), BH-FDR across the displayed grid:
#   LEFT  = composition: logistic  I(cell in state) ~ combo   -> beta = log-odds (combo vs NTC)
#   RIGHT = within-state Doxo1: OLS  Doxo1 ~ combo            -> beta = Δ Doxo1 (combo vs NTC)
# Colour = the regression beta; * = FDR<FDR. A cell that can't be fit (within-state
# <MIN_N combo cells, or logistic separation) is greyed and marked "·". NEUROG1+SIM1 boxed.
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

MIN_N = 3
obs = adata.obs
state = obs[fu.CELL_STATE_COL].astype(str).values
pert = obs["perturbation_clean"].astype(str).values
tpv = obs["time_point"].astype(str).values
score = obs["Doxo1program_score"].astype(float).values
order = fu.STATE_ORDER

def _one_sided(beta, p_two):
    return p_two / 2 if beta > 0 else 1 - p_two / 2

for DAY in fu.TIMEPOINT_ORDER:
    combos = sorted(e1_combos(DAY))
    dmask = tpv == DAY
    ntc = dmask & (pert == fu.NTC_LABEL)
    rows = []
    for p in combos:
        pm = dmask & (pert == p); npm = int(pm.sum())
        for s in order:
            in_s = state == s
            a = int((pm & in_s).sum())
            rec = {"pair": p, "state": s, "n_combo": a,
                   "beta_comp": np.nan, "p_comp": np.nan, "beta_dox": np.nan, "p_dox": np.nan}
            # composition: logistic in_state ~ combo (combo + NTC cells)
            g = pm | ntc
            dC = pd.DataFrame({"in_s": in_s[g].astype(float), "combo": pm[g].astype(float)})
            try:
                mc = smf.glm("in_s ~ combo", dC, family=sm.families.Binomial()).fit()
                b, se, p2 = float(mc.params["combo"]), float(mc.bse["combo"]), float(mc.pvalues["combo"])
                if np.isfinite(b) and np.isfinite(se) and se < 50:
                    rec["beta_comp"] = float(np.clip(b, -6, 6)); rec["p_comp"] = _one_sided(b, p2)
            except Exception:
                pass
            # within-state Doxo1: OLS Doxo1 ~ combo (combo + NTC cells in the state)
            x, y = score[pm & in_s], score[ntc & in_s]
            if len(x) >= MIN_N and len(y) >= MIN_N:
                dD = pd.DataFrame({"y": np.r_[x, y], "combo": np.r_[np.ones(len(x)), np.zeros(len(y))]})
                md = smf.ols("y ~ combo", dD).fit()
                b, p2 = float(md.params["combo"]), float(md.pvalues["combo"])
                rec["beta_dox"] = b; rec["p_dox"] = _one_sided(b, p2)
            rows.append(rec)
    r = pd.DataFrame(rows)
    for pcol, fcol in [("p_comp", "fdr_comp"), ("p_dox", "fdr_dox")]:
        r[fcol] = np.nan; ok = r[pcol].notna()
        if ok.any():
            r.loc[ok, fcol] = multipletests(r.loc[ok, pcol], method="fdr_bh")[1]
    r.to_csv(fu.FIG_DIR / f"Fig3_state_vs_doxo1_{DAY}.csv", index=False)
    print(f"{DAY}: {len(r)} cells | composition sig {(r.fdr_comp < FDR).sum()} "
          f"(untestable {r.p_comp.isna().sum()}), within-state sig {(r.fdr_dox < FDR).sum()} "
          f"(untestable {r.p_dox.isna().sum()})")

    def pv(col):
        return r.pivot_table(index="pair", columns="state", values=col).reindex(index=combos, columns=order)
    comp, comp_fdr = pv("beta_comp"), pv("fdr_comp")
    dox, dox_fdr = pv("beta_dox"), pv("fdr_dox")
    ordr = comp[fu.DIFFERENTIATED_STATES].max(axis=1).sort_values(ascending=False).index
    comp, comp_fdr, dox, dox_fdr = (m.reindex(ordr) for m in (comp, comp_fdr, dox, dox_fdr))

    def ann(fdr):
        A = np.where(fdr.values < FDR, "*", "").astype(object)
        A[np.isnan(fdr.values)] = "·"               # tested-nonsig = "", untestable = "·"
        return A

    fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, max(3, len(ordr) * 0.34 + 1.5)))
    for ax, mat, fdr, cmap, lab, ttl in [
            (axL, comp, comp_fdr, "PuOr_r", "log-odds (combo in state)",
             f"Composition shift — logistic β\n(combo in state vs NTC; * FDR<{FDR}, · unfit)"),
            (axR, dox, dox_fdr, "RdBu_r", "Δ Doxo1 (OLS β)",
             f"Within-state Doxo1 — OLS β\n(combo > NTC within state; * FDR<{FDR}, · n<{MIN_N})")]:
        ax.set_facecolor("0.85")                    # untestable (masked) cells show grey
        v = np.nanmax(np.abs(mat.values)) or 1.0
        sns.heatmap(mat, cmap=cmap, center=0, vmin=-v, vmax=v, annot=ann(fdr), fmt="",
                    linewidths=0.4, linecolor="white", cbar_kws={"label": lab}, ax=ax)
        ax.set_title(ttl, fontsize=9); ax.set_xlabel("")
        plt.setp(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
    axL.set_ylabel("Double-KO pair"); axR.set_ylabel(""); axR.set_yticks([])
    if FORCE in list(ordr):
        yi = list(ordr).index(FORCE)
        for ax in (axL, axR):
            ax.add_patch(plt.Rectangle((0, yi), len(order), 1, fill=False, edgecolor="black", lw=2.2))
    fig.suptitle(f"Perturbation → cell state → Doxo1 — {DAY}  "
                 f"(statsmodels regression; FDR over the displayed grid)", fontsize=11)
    fu.savefig(f"Fig3_state_vs_doxo1_{DAY}", fig)